# FDA Adverse Events — Power BI Data Export

This notebook exports clean, aggregated CSVs that feed directly into Power BI.
Run all cells top to bottom. You will get 5 CSV files ready to import.

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('fda_adverse_events_2015_2026_CLEAN.csv', low_memory=False)
df['serious_binary'] = df['serious'].map({'Yes': 1, 'No': 0})

print(f'Loaded: {df.shape[0]:,} rows')

Loaded: 528,000 rows


---
### Export 1: Serious Event Rate by Age Group

In [3]:
age_order = ['Infant(0-2)', 'Child(3-12)', 'Teen(13-18)', 'Adult(19-40)',
             'Middle-Aged(41-65)', 'Senior(66-80)', 'Elderly(81+)']

age_df = df.groupby('age_group').agg(
    Total_Reports=('serious_binary', 'count'),
    Serious_Count=('serious_binary', 'sum')
).reset_index()

age_df['Serious_Rate_Pct'] = (age_df['Serious_Count'] / age_df['Total_Reports'] * 100).round(2)
age_df.columns = ['Age_Group', 'Total_Reports', 'Serious_Count', 'Serious_Rate_Pct']
age_df['Sort_Order'] = age_df['Age_Group'].map({v: i for i, v in enumerate(age_order)})
age_df = age_df.sort_values('Sort_Order').drop(columns='Sort_Order')

age_df.to_csv('pbi_serious_by_age.csv', index=False)
print('Saved: pbi_serious_by_age.csv')
age_df

Saved: pbi_serious_by_age.csv


,Age_Group,Total_Reports,Serious_Count,Serious_Rate_Pct
3,Infant(0-2),3101,2800,90.29
1,Child(3-12),9767,6623,67.81
6,Teen(13-18),8617,6253,72.57
0,Adult(19-40),53277,42364,79.52
4,Middle-Aged(41-65),166278,130198,78.30
5,Senior(66-80),108781,86065,79.12
2,Elderly(81+),25842,20945,81.05
7,Unknown,152337,99752,65.48


---
### Export 2: Serious Event Rate by Drug Count (Polypharmacy)

In [5]:
poly_order = ['Single', '2-3 Drugs', '4-5 Drugs', 'Polypharmacy(6+)', 'Unknown']

poly_df = df.groupby('drug_count_category').agg(
    Total_Reports=('serious_binary', 'count'),
    Serious_Count=('serious_binary', 'sum')
).reset_index()

poly_df['Serious_Rate_Pct'] = (poly_df['Serious_Count'] / poly_df['Total_Reports'] * 100).round(2)
poly_df.columns = ['Drug_Count_Category', 'Total_Reports', 'Serious_Count', 'Serious_Rate_Pct']
poly_df['Sort_Order'] = poly_df['Drug_Count_Category'].map({v: i for i, v in enumerate(poly_order)})
poly_df = poly_df.sort_values('Sort_Order').drop(columns='Sort_Order')

poly_df.to_csv('pbi_polypharmacy.csv', index=False)
print('Saved: pbi_polypharmacy.csv')
poly_df

Saved: pbi_polypharmacy.csv


,Drug_Count_Category,Total_Reports,Serious_Count,Serious_Rate_Pct
3,Single,106633,62711,58.81
0,2-3 Drugs,126066,84784,67.25
1,4-5 Drugs,71972,55851,77.60
2,Polypharmacy(6+),221528,189857,85.70
4,Unknown,1801,1797,99.78


---
### Export 3: Top 15 Drugs by Report Volume

In [7]:
top_drugs_df = df['suspect_drug'].value_counts().head(15).reset_index()
top_drugs_df.columns = ['Drug_Name', 'Report_Count']

top_drugs_df.to_csv('pbi_top_drugs.csv', index=False)
print('Saved: pbi_top_drugs.csv')
top_drugs_df

Saved: pbi_top_drugs.csv


,Drug_Name,Report_Count
0,TOFACITINIB,13807
1,RISPERIDONE,13487
2,RIVAROXABAN,12968
3,AVANDIA,10091
4,ETANERCEPT,8751
5,DUPILUMAB,8246
6,ADALIMUMAB,8131
7,SODIUM OXYBATE,8008
8,VEDOLIZUMAB,7875
9,PREGABALIN,7845


---
### Export 4: Adverse Event Trends Over Time

In [9]:
trends_df = df.groupby('year').agg(
    Total_Reports=('report_id', 'count'),
    Serious_Count=('serious_binary', 'sum')
).reset_index()

trends_df['Serious_Rate_Pct'] = (trends_df['Serious_Count'] / trends_df['Total_Reports'] * 100).round(2)
trends_df['Non_Serious_Count'] = trends_df['Total_Reports'] - trends_df['Serious_Count']
trends_df.columns = ['Year', 'Total_Reports', 'Serious_Count', 'Serious_Rate_Pct', 'Non_Serious_Count']

trends_df.to_csv('pbi_trends_over_time.csv', index=False)
print('Saved: pbi_trends_over_time.csv')
trends_df

Saved: pbi_trends_over_time.csv


,Year,Total_Reports,Serious_Count,Serious_Rate_Pct,Non_Serious_Count
0,2015,48000,37555,78.24,10445
1,2016,48000,34481,71.84,13519
2,2017,48000,32832,68.40,15168
3,2018,48000,34449,71.77,13551
4,2019,48000,33340,69.46,14660
5,2020,48000,39054,81.36,8946
6,2021,48000,36448,75.93,11552
7,2022,48000,36431,75.90,11569
8,2023,48000,37250,77.60,10750
9,2024,48000,33802,70.42,14198


---
### Export 5: Outcome Type Breakdown

In [11]:
outcome_cols = ['is_fatal', 'is_hospitalized', 'is_life_threat', 'is_disabling']
labels = ['Fatal', 'Hospitalized', 'Life-Threatening', 'Disabling']

outcome_df = pd.DataFrame({
    'Outcome_Type': labels,
    'Count': [df[col].sum() for col in outcome_cols],
    'Percent_of_All_Reports': [(df[col].sum() / len(df) * 100).round(2) for col in outcome_cols]
})

outcome_df.to_csv('pbi_outcome_types.csv', index=False)
print('Saved: pbi_outcome_types.csv')
outcome_df

Saved: pbi_outcome_types.csv


,Outcome_Type,Count,Percent_of_All_Reports
0,Fatal,54301,10.28
1,Hospitalized,188042,35.61
2,Life-Threatening,22871,4.33
3,Disabling,13524,2.56


---
### Export 6: KPI Summary Card Data

In [13]:
kpi_df = pd.DataFrame([
    {'KPI': 'Total Reports',          'Value': len(df),                                          'Display': f"{len(df):,}"},
    {'KPI': 'Serious Events',         'Value': int(df['serious_binary'].sum()),                   'Display': f"{int(df['serious_binary'].sum()):,}"},
    {'KPI': 'Serious Event Rate',     'Value': round(df['serious_binary'].mean() * 100, 1),       'Display': f"{round(df['serious_binary'].mean()*100,1)}%"},
    {'KPI': 'Fatal Events',           'Value': int(df['is_fatal'].sum()),                         'Display': f"{int(df['is_fatal'].sum()):,}"},
    {'KPI': 'Hospitalized Events',    'Value': int(df['is_hospitalized'].sum()),                  'Display': f"{int(df['is_hospitalized'].sum()):,}"},
    {'KPI': 'Years of Data',          'Value': df['year'].nunique(),                              'Display': f"{df['year'].nunique()} years"},
    {'KPI': 'Unique Drugs Tracked',   'Value': df['suspect_drug'].nunique(),                      'Display': f"{df['suspect_drug'].nunique():,}"},
    {'KPI': 'Countries Represented',  'Value': df['country'].nunique(),                           'Display': f"{df['country'].nunique():,}"},
])

kpi_df.to_csv('pbi_kpis.csv', index=False)
print('Saved: pbi_kpis.csv')
kpi_df

Saved: pbi_kpis.csv


,KPI,Value,Display
0,Total Reports,528000.0,"528,000"
1,Serious Events,395000.0,"395,000"
2,Serious Event Rate,74.8,74.8%
3,Fatal Events,54301.0,"54,301"
4,Hospitalized Events,188042.0,"188,042"
5,Years of Data,11.0,11 years
6,Unique Drugs Tracked,9828.0,"9,828"
7,Countries Represented,162.0,162


In [15]:
print('\n✅ All exports complete. Files ready for Power BI:')
files = [
    ('pbi_serious_by_age.csv',    'Serious rate by age group → Bar chart'),
    ('pbi_polypharmacy.csv',      'Polypharmacy serious rate → Bar chart'),
    ('pbi_top_drugs.csv',         'Top 15 drugs by volume → Horizontal bar'),
    ('pbi_trends_over_time.csv',  'Reports & serious rate by year → Line chart'),
    ('pbi_outcome_types.csv',     'Outcome type breakdown → Donut chart'),
    ('pbi_kpis.csv',              'Summary numbers → KPI cards'),
]
for f, desc in files:
    print(f'  {f:40s} → {desc}')


✅ All exports complete. Files ready for Power BI:
  pbi_serious_by_age.csv                   → Serious rate by age group → Bar chart
  pbi_polypharmacy.csv                     → Polypharmacy serious rate → Bar chart
  pbi_top_drugs.csv                        → Top 15 drugs by volume → Horizontal bar
  pbi_trends_over_time.csv                 → Reports & serious rate by year → Line chart
  pbi_outcome_types.csv                    → Outcome type breakdown → Donut chart
  pbi_kpis.csv                             → Summary numbers → KPI cards
